# Notebook #1: Baseline assessment

## ODSC East 2025: VCPO1 on Llama Code Demo

In this notebook, we'll:

* Load in the baseline ~8B parameter Llama 3.1 model
* Load in the GSM8K dataset
* Generate responses using the baseline model, saving the results to S3 along the way

# Imports, installation, and setup

In [1]:
# this cell should take ~15 seconds
# import boto3 Removed because it is AWS specific and not needed for the current task
from datetime import datetime
#import getpass Removed because it is AWS specific.
import json
import os
from os import path
import random
import re
import string
import typing
import datasets
import huggingface_hub
import torch
#import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

In [2]:
#AWS_ACCESS_KEY_ID = getpass.getpass("Enter your AWS_ACCESS_KEY_ID: ")
#AWS_SECRET_ACCESS_KEY = getpass.getpass("Enter your AWS_SECRET_ACCESS_KEY: ")

#os.environ["AWS_ACCESS_KEY_ID"] = AWS_ACCESS_KEY_ID
#os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_ACCESS_KEY

#S3_BUCKET_NAME = "data-science-talks"
#S3_CLIENT = boto3.client("s3")

#JSONable = typing.Union[dict, list]

In [4]:
if not path.exists('data'):
    os.mkdir("data")

In [5]:
# authentication required to read in the Llama model
huggingface_hub.login()

# IO functions

In [6]:
JSONable = typing.Union[dict, list]

def save_jsonable_object_locally(obj: JSONable, filename: str, save_dir: str = "data") -> None:
    os.makedirs(save_dir, exist_ok=True)
    filepath = os.path.join(save_dir, f"{filename}.json")
    print(f"Saving object locally to {filepath}")
    with open(filepath, "w") as f:
        json.dump(obj, f, indent=2)

def load_jsonable_object_locally(filename: str, load_dir: str = "data") -> JSONable:
    filepath = os.path.join(load_dir, f"{filename}.json")
    print(f"Reading {filepath} into Python")
    with open(filepath, "r") as f:
        return json.load(f)

# Reading in the LLama model and GSM8K dataset

In [7]:
MAX_SEQ_LENGTH = 256
MODEL_NAME = "meta-llama/meta-Llama-3.1-8B-Instruct"
DATASET_NAME = 'openai/gsm8k'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16) #max_length=MAX_SEQ_LENGTH, torch_dtype=torch.bfloat16

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [6]:
gsm8k_train_data = datasets.load_dataset(DATASET_NAME, 'main')["train"]

Script to get the lengths of the gold answers:

```python
import pandas as pd


gsm8k_data = datasets.load_dataset('openai/gsm8k', 'main')
test_set = gsm8k_data['test']
test_set_shuffled = test_set.shuffle(seed=250217)

subset = test_set_shuffled.select(range(100))

gold_lens = []
for i, sample in enumerate(subset):
    gold_raw = sample["answer"]
    gold_answer = _extract_hash_answer(gold_raw)

    # Tokenize gold_answer
    tok = tokenizer(gold_raw, return_tensors="pt", padding=False, truncation=False)

    # Count the tokens
    num_tokens = len(tok["input_ids"][0])

    gold_lens.append(
        {"index": i, "gold_answer": gold_answer, "token_length": num_tokens}
    )

# Convert to a Pandas DataFrame
df = pd.DataFrame(gold_lens)
print(df.head(30))
```

This was used to confirm that the lengths of the answers fit within the 256 character
`MAX_SEQ_LENGTH`.

# Checking baseline performance

The code below allows one to generate the objects that can be used to assess what proportion of the
GSM8K test set the Llama 8B Instruct model gets right.

In [10]:
def _is_valid_number(candidate: str) -> bool:
    """
    Check if 'candidate' matches one of the patterns:
    - plain integer (e.g. '39')
    - dollar + integer (e.g. '$39')
    - comma-separated integer (e.g. '43,500')
    """
    candidate = candidate.strip()
    # e.g. remove leading/trailing '$', but handle the case if it starts with $
    # We'll use a regex approach to handle commas or $ sign:
    pattern = r"^\$?\d{1,3}(,\d{3})*(\.\d+)?$"
    return bool(re.match(pattern, candidate))


# from 
# https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb
def _extract_xml_answer(text: str) -> str:
    answer = text.split("<answer>")[-1]
    answer = answer.split("</answer>")[0]
    return answer.strip()


def _extract_hash_answer(text: str) -> str:
    return text.split("####")[1].strip()


def _random_three_alphanumeric():
    # 'ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789'
    chars = string.ascii_letters + string.digits
    return ''.join(random.choices(chars, k=3))


def generate_answers(
    model,
    tokenizer,
    system_prompt: str,
    dataset_subset,
    local_prefix: str,
    num_attempts: int = 5,
    max_length_prompt: int = 256,
    max_new_tokens: int = 256,
    start_index: int = 0,
    end_index: typing.Optional[int] = None,
):
    """
    Generate multiple answers for each sample in a dataset subset using a specified model,
    and upload the results to S3.

    Args:
        model: The model to use for generation (e.g., baseline model or fine-tuned LoRA model).
        tokenizer: The tokenizer corresponding to the model.
        system_prompt: The system prompt to prepend to each question.
        dataset_subset: The dataset subset to process (e.g., 100 shuffled GSM8K test samples).
        local_prefix: Prefix for local file paths(e.g., 'gsm8k_llama_7b_100_record_test_').
        num_attempts: Number of answer attempts per sample (default: 5).
        max_length_prompt: Maximum token length for the input prompt (default: 256).
        max_new_tokens: Maximum new tokens to generate (default: 256).
        start_index: Index to start processing from (default: 1, useful for resuming).
    """
    # Set model to evaluation mode
    model.eval()

    # Move model to CUDA
    model.to("cuda")

    # Handle index bounds
    dataset_len = len(dataset_subset)
    start_index = max(1, start_index)
    end_index = dataset_len if end_index is None else min(dataset_len, end_index)

    # Process each sample in the specified range
    for i in range(start_index, end_index):
        sample = dataset_subset[i - 1]

        # Extract question and gold answer from the sample
        question_text = sample["question"]
        gold_raw = sample["answer"]
        gold_answer = _extract_hash_answer(gold_raw)

        # Construct the prompt based on use_chat_template
        messages = [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': question_text},
        ]
        full_prompt_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

        # Tokenize the prompt
        inputs = tokenizer(
            full_prompt_text,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length_prompt,
        ).to("cuda")

        # Get the length of the prompt tokens
        prompt_len = inputs["input_ids"].shape[1]

        # Generate `num_attempts` answers for the sample
        sampled_answers = []
        for _ in range(num_attempts):
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=True,
                    temperature=0.8,
                    top_p=0.95,
                )

            # Extract newly generated tokens (after the prompt)
            gen_tokens = outputs[0, prompt_len:]
            completion_text = tokenizer.decode(gen_tokens, skip_special_tokens=True)

            # Extract and validate the predicted answer
            predicted_answer = _extract_xml_answer(completion_text)
            if _is_valid_number(predicted_answer):
                pred_clean = predicted_answer.replace("$", "").replace(",", "").strip()
                gold_clean = gold_answer.replace("$", "").replace(",", "").strip()
                correct = pred_clean == gold_clean
            else:
                correct = False

            # Store attempt details
            sampled_answers.append(
                {
                    "raw_text": completion_text,
                    "predicted_answer": predicted_answer,
                    "valid_number": _is_valid_number(predicted_answer),
                    "correct": correct,
                }
            )

        # Compile the record for this sample
        record = {
            "index": i,
            "question": question_text,
            "gold_answer": gold_answer,
            "model_outputs": sampled_answers,
        }

        now = datetime.now()

        timestamp_str = now.strftime("%b-%d-%Y_%I-%M-%S%p")

        filename = f"{_random_three_alphanumeric()}_{timestamp_str}"
        # Upload to S3 with a 1-based index in the key
        save_jsonable_object_locally(
            record, filename, f'{local_prefix}'
        )


In [ ]:
gsm8k_data = datasets.load_dataset(DATASET_NAME, 'main')
test_set = gsm8k_data['test']
test_set_shuffled = test_set.shuffle(seed=250217)

SYSTEM_PROMPT = """
### EXAMPLE ###
Q: 3+2
<reasoning>
3 plus 2 is 5
</reasoning>
<answer>
5
</answer>

Now follow the same format EXACTLY for each question:

<reasoning>
...
</reasoning>
<answer>
...
</answer>
"""

PREFIX_NEW = "./data/1-Baseline/"

generate_answers(
    model=model,
    tokenizer=tokenizer,
    system_prompt=SYSTEM_PROMPT,
    dataset_subset=test_set_shuffled,
    local_prefix=PREFIX_NEW,
)

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Saving object locally to ./data/1-Baseline/b7a_Jun-05-2026_12-30-00PM.json


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Saving object locally to ./data/1-Baseline/BuT_Jun-05-2026_12-30-12PM.json


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Saving object locally to ./data/1-Baseline/Qpu_Jun-05-2026_03-01-41PM.json


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Saving object locally to ./data/1-Baseline/YAe_Jun-05-2026_03-01-50PM.json


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Code snippet to read in and display one of the objects from S3:

```python
import textwrap


def _print_wrapping_as_list_of_lines(data: dict) -> str:
    # Wrap 'question' as a list of lines
    if "question" in data and data["question"]:
        question_wrapped = textwrap.wrap(data["question"], width=89)
        data["question"] = question_wrapped

    # Wrap 'raw_text' as a list of lines
    for entry in data.get("model_outputs", []):
        if "raw_text" in entry and entry["raw_text"]:
            wrapped_lines = textwrap.wrap(entry["raw_text"], width=89)
            entry["raw_text"] = wrapped_lines

    # Return as pretty-printed JSON
    return json.dumps(data, indent=2)

data = download_object_from_s3("gsm8k_llama_7b_100_record_test_1")
s = _print_wrapping_as_list_of_lines(data)
print(s)
```